In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')


if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key found and looks good so far!")

openai = OpenAI()

MODELS = ["meta-llama/llama-3.1-8b-instruct", "mistralai/mistral-nemo-12b-instruct-v1", "qwen/qwen-2.5-7b-instruct"]
	
def generate_dataset(topic, model_name, count):
    prompt = f'Generate {count} JSON objects for a dataset about "{topic}". Structure: {{"input": "...", "output": "..."}}'
    json_schema = {
        "name": "dataset_schema",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "samples": {
                    "type": "array",
                    "minItems": int(count),
                    "maxItems": int(count),
                    "items": {
                        "type": "object",
                        "properties": {
                            "input": {"type": "string"},
                            "output": {"type": "string"}
                        },
                        "required": ["input", "output"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["samples"],
            "additionalProperties": False
        }
    }
    response = openai.chat.completions.create(
        model=model_name,
        messages=[{"role": "system", "content": "You are a data engineer. Output only raw JSON."}, {"role": "user", "content": prompt}],
        response_format={
            "type": "json_schema",
            "json_schema": json_schema
        }
    )
    return response.choices[0].message.content

def save_to_file(json_data):
    with open("dataset.json", "w") as f:
        f.write(json_data)
    return "dataset.json"

def orchestrate_generation(topic, model_name, count):
    raw_json = generate_dataset(topic, model_name, count)
    file_path = save_to_file(raw_json)
    return raw_json, file_path

with gr.Blocks() as synthesis:
    gr.Markdown("Synthesis")
    
    with gr.Row():
        topic_in = gr.Textbox(label="Dataset Topic")
        model_drop = gr.Dropdown(MODELS, label="Select Open Source Model")
        count_slider = gr.Slider(1, 10, value=3, label="Sample Count", step=1)
    
    gen_btn = gr.Button("Generate Dataset")
    json_out = gr.JSON(label="Preview")
    file_out = gr.File(label="Download Dataset")
    
    gen_btn.click(orchestrate_generation, inputs=[topic_in, model_drop, count_slider], outputs=[json_out, file_out])

synthesis.launch(inbrowser=True)

API key found and looks good so far!
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
